&nbsp;

## No Proximal CRE transcripts analysis

- public transcriptome (bulk RNA-sequencing data) で選抜した transcripts と、[`fanta.bio`](https://fanta.bio/) を対応付けたとき、TSS ± 500 bp に CRE がない transcript がどのような状態なのかを調べるための notebook
- 粕川先生から、gene level で 10kb までの距離では紐づく gene - cre 対応データをいただいた
- これを元に、reference chromosome 以外の transcript, 10kb 以内に紐づかない transcript (かつ gene level でも紐づかない)、10kb 以内であれば、紐づく transcript の 3つにわける

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt

from IPython.display import display


pl.Config.set_fmt_str_lengths(100)
os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## `fanta.bio` (ver.1.2.0) gene level CRE association

- 粕川先生から、gene - CRE level relationship file をいただいた
- これは、ver.1.2.0 に該当する
- 10 kb までの対応

In [2]:
# https://data.fanta.bio/cre/v1.2.1/00README_v1.2.1.txt
fantabio_bed = pl.read_csv(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    has_header=False,
    new_columns=[
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "cre_id",
        "score",
        "strand",
        "thickStart",
        "thickEnd",
        "itemRgb",
        "cre_name",
        "pooled_cre_scores"
    ]
).with_columns(
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
).select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class"
)

display(fantabio_bed)

cre_id,cre_name,fantabio_cre_class
str,str,str
"""FCHS_1""","""cp1@DDX11L1""","""PLA"""
"""FCHS_2""","""cp1@WASH7P""","""ELA"""
"""FCHS_447416""","""cp1@DDX11L17""","""ELA"""
"""FCHS_447417""","""FCHS_447417""","""ELA"""
"""FCHS_3""","""cp2@LOC101928626""","""ELA"""
…,…,…
"""FCHS_447311""","""FCHS_447311""","""ELA"""
"""FCHS_447312""","""FCHS_447312""","""ELA"""
"""FCHS_447313""","""FCHS_447313""","""ELA"""


In [3]:
human_gene_cre_association = pl.read_csv(
    "../../Data/Database/fantabio/gene_neighbor/human_gene_cre_associate.csv.gz",
    separator=",",
    n_threads=16
).sort("distance").join(
    fantabio_bed,
    on="cre_id",
    how="inner"
)

display(human_gene_cre_association)

gene_id,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,str,i64,str,str,str,str
"""ENSG00000310526.1""","""FCHS_2""",0,"""human""","""hg38""","""cp1@WASH7P""","""ELA"""
"""ENSG00000243485.6""","""FCHS_2""",0,"""human""","""hg38""","""cp1@WASH7P""","""ELA"""
"""ENSG00000310528.1""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
"""ENSG00000308415.1""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
"""ENSG00000241860.8""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
…,…,…,…,…,…,…
"""ENSG00000298375.1""","""FCHS_253394""",10000,"""human""","""hg38""","""FCHS_253394""","""ELA"""
"""ENSG00000232687.1""","""FCHS_533966""",10000,"""human""","""hg38""","""FCHS_533966""","""ELA"""
"""ENSG00000302188.1""","""FCHS_413906""",10000,"""human""","""hg38""","""FCHS_413906""","""ELA"""


In [4]:
# gene level で CRE と紐づいている gene id を取得
cre_gene_ids = human_gene_cre_association.select(
    pl.col("gene_id").unique().implode()
).to_series()

&nbsp;

&nbsp;

&nbsp;

## GENCODE release 49 GTF file

- `fanta.bio` の CRE に対応しなかった transcript が おそらく haplotype 由来の transcript が含まれている可能性があるため、GTF ファイルの情報から取得する

In [5]:
# For more information, see https://www.gencodegenes.org/pages/biotypes.html
gencode_gtf = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf",
    separator='\t',
    has_header=False,
    comment_prefix='#',
    new_columns=['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attributes'],
    n_threads=16
).filter(
    pl.col('feature') == 'transcript'
).with_columns(
    # Unique keys: extract the first (and only) quoted value
    pl.col("attributes").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_id"),
    pl.col("attributes").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id"),
    pl.col("attributes").str.extract(r'transcript_type "([^"]+)"', 1).alias("transcript_type"),
    # tag can appear multiple times; extract_all returns full matches like tag "basic"
    pl.col("attributes")
        .str.extract_all(r'tag "[^"]+"')
        .list.eval(pl.element().str.extract(r'"([^"]+)"', 1))
        .alias("tags")
).with_columns(
    pl.col("tags").list.n_unique().alias("number_of_tags")
).unique(
    subset=["transcript_id", "transcript_type", "gene_id", "tags", "number_of_tags", "strand"]
).select(
    pl.col("transcript_id"),
    pl.col("seqname"),
    pl.col("transcript_type"),
    pl.col("gene_id"),
    pl.col("tags"),
    pl.col("number_of_tags"),
    pl.col("strand"),
)

display(gencode_gtf)

transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand
str,str,str,str,list[str],u32,str
"""ENST00000587953.5""","""chr18""","""protein_coding_CDS_not_defined""","""ENSG00000101746.17""","[""mRNA_end_NF""]",1,"""-"""
"""ENST00000533211.6""","""chr11""","""protein_coding""","""ENSG00000173898.17""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]",6,"""-"""
"""ENST00000661181.1""","""chr6""","""lncRNA""","""ENSG00000286299.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]",4,"""+"""
"""ENST00000818388.1""","""chr15""","""lncRNA""","""ENSG00000290824.2""","[""TAGENE""]",1,"""-"""
"""ENST00000957464.1""","""chr14""","""protein_coding""","""ENSG00000133961.23""","[""basic"", ""TAGENE""]",2,"""-"""
…,…,…,…,…,…,…
"""ENST00000901146.1""","""chr14""","""protein_coding""","""ENSG00000119636.17""","[""basic"", ""GENCODE_Primary"", ""TAGENE""]",3,"""+"""
"""ENST00000289749.6""","""chr1""","""protein_coding""","""ENSG00000158747.16""","[""basic"", ""CCDS""]",2,"""+"""
"""ENST00000912662.1""","""chr16""","""protein_coding""","""ENSG00000166848.8""","[""basic"", ""GENCODE_Primary"", ""TAGENE""]",3,"""+"""


&nbsp;

&nbsp;

&nbsp;

## down5p no proximal CRE

In [6]:
down5p_no_proximal_cre_transcripts = pl.read_parquet(
    "../../out/fantabio/no_proximal_cre_transcirpt_list/bottom5p_transcript_no_cre_v261001.parquet"
)

print(f"up5p no proximal cre - transcripts: {down5p_no_proximal_cre_transcripts.select('transcript_id').n_unique()}")
display(down5p_no_proximal_cre_transcripts)

up5p no proximal cre - transcripts: 4458


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000535641.5""",84,132,190,-48,58,64,129,-6,"""ANAPC5""","""HGNC:15713""","""retained_intron""","""ENSG00000089053.14""",[],"""-""",0,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000510365.5""",64,112,230,-48,44,53,154,-9,"""ACOX3""","""HGNC:121""","""retained_intron""","""ENSG00000087008.17""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000831501.1""",44,92,270,-48,33,47,171,-14,null,null,"""lncRNA""","""ENSG00000310527.1""","[""TAGENE""]","""-""",1,null,null,null,"""havana_homo_sapiens""","[""CLS3 project"", ""overlaps pseudogene""]",2
"""ENST00000589589.1""",51,99,256,-48,37,53,161,-16,"""PDCD2L""","""HGNC:28194""","""retained_intron""","""ENSG00000126249.8""",[],"""+""",0,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000623575.4""",44,92,270,-48,24,40,187,-16,null,null,"""lncRNA""","""ENSG00000275496.5""","[""TAGENE""]","""-""",1,"""URS0002A5A522""","""ENSEMBL_GENCODE""",null,"""ensembl_havana_lincrna_homo_sapiens""","[""CLS3 project""]",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000559522.1""",17,264,125,-247,5,166,80,-161,"""PIF1""","""HGNC:26220""","""retained_intron""","""ENSG00000140451.14""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000413097.6""",21,268,117,-247,11,183,57,-172,"""PARP6""","""HGNC:26921""","""protein_coding_CDS_not_defined""","""ENSG00000137817.19""",[],"""-""",0,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000474699.1""",16,264,126,-248,12,174,65,-162,"""TIA1""","""HGNC:11802""","""retained_intron""","""ENSG00000116001.18""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0


In [7]:
# gencode_gtf と up5p_no_proximal_cre_transcripts を join
down5p_no_proximal_cre_transcripts_join = (
    gencode_gtf.join(
        down5p_no_proximal_cre_transcripts,
        on=["transcript_id", "transcript_type", "gene_id", "tags", "number_of_tags", "strand"],
        how="inner"
    )
).sort(
    "HN2",
    descending=True
)

pl.Config.set_tbl_rows(114)
print(down5p_no_proximal_cre_transcripts_join.select("seqname").sort("seqname").unique())

pl.Config.set_tbl_rows(4)
display(down5p_no_proximal_cre_transcripts_join)

shape: (124, 1)
┌────────────┐
│ seqname    │
│ ---        │
│ str        │
╞════════════╡
│ GL000194.1 │
│ GL000195.1 │
│ GL000205.2 │
│ GL000214.1 │
│ GL000220.1 │
│ GL000221.1 │
│ GL000250.2 │
│ GL000251.2 │
│ GL000252.2 │
│ GL000253.2 │
│ GL000254.2 │
│ GL000255.2 │
│ GL000256.2 │
│ GL000258.2 │
│ GL339449.2 │
│ GL383519.1 │
│ GL383522.1 │
│ GL383527.1 │
│ GL383534.2 │
│ GL383550.2 │
│ GL383563.3 │
│ GL383574.1 │
│ GL383580.2 │
│ GL383581.2 │
│ GL877876.1 │
│ GL949752.1 │
│ KI270711.1 │
│ KI270713.1 │
│ KI270714.1 │
│ KI270733.1 │
│ KI270734.1 │
│ KI270744.1 │
│ KI270762.1 │
│ KI270763.1 │
│ KI270769.1 │
│ KI270777.1 │
│ KI270779.1 │
│ KI270791.1 │
│ KI270804.1 │
│ KI270806.1 │
│ KI270809.1 │
│ KI270815.1 │
│ KI270816.1 │
│ KI270819.1 │
│ KI270831.1 │
│ KI270832.1 │
│ KI270844.1 │
│ KI270847.1 │
│ KI270848.1 │
│ KI270850.1 │
│ KI270853.1 │
│ KI270854.1 │
│ KI270855.1 │
│ KI270857.1 │
│ KI270860.1 │
│ KI270861.1 │
│ KI270873.1 │
│ …          │
│ KN196481.1 │
│ KN538369.1 │
│ KQ03138

transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32
"""ENST00000482001.1""","""chr9""","""protein_coding_CDS_not_defined""","""ENSG00000148218.17""",[],0,"""-""",80,128,198,-48,48,67,136,-19,"""ALAD""","""HGNC:395""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000609771.1""","""chr4""","""lncRNA""","""ENSG00000272885.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-""",79,127,200,-48,42,80,129,-38,null,null,"""URS00005B55C8""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000497231.7""","""chr5""","""retained_intron""","""ENSG00000164904.19""",[],0,"""-""",15,266,125,-251,10,182,59,-172,"""ALDH7A1""","""HGNC:877""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Filter non-reference chromosome (haplotypes, scaffold...)

- ここでは、reference chromosome 以外から由来する transcript を選抜する

In [8]:
down5p_no_proximal_cre_transcripts_join_other_chr_check = down5p_no_proximal_cre_transcripts_join.filter(
    ~pl.col("seqname").str.contains("chr")
).with_columns(
    has_cre_gene=pl.col("gene_id").is_in(cre_gene_ids)
).with_columns(
    pl.lit("other chromosome derived").alias("flag")
)

print(down5p_no_proximal_cre_transcripts_join_other_chr_check.select("transcript_id").n_unique())
display(down5p_no_proximal_cre_transcripts_join_other_chr_check)

254


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,bool,str
"""ENST00000709866.1""","""MU273340.1""","""nonsense_mediated_decay""","""ENSG00000292155.1""","[""not_organism_supported"", ""mRNA_start_NF"", ""cds_start_NF""]",3,"""-""",71,119,216,-48,47,81,123,-34,"""RETSAT""","""HGNC:25991""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""not organism-supported"", ""QC splicing correct""]",2,false,"""other chromosome derived"""
"""ENST00000631722.1""","""GL383563.3""","""retained_intron""","""ENSG00000262334.6""",[],0,"""-""",50,98,258,-48,29,53,169,-24,"""RPH3AL""","""HGNC:10296""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,false,"""other chromosome derived"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000797527.1""","""GL000195.1""","""lncRNA""","""ENSG00000303853.1""","[""TAGENE""]",1,"""-""",26,226,154,-200,12,157,82,-145,null,null,"""URS0002A30CE3""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,false,"""other chromosome derived"""
"""ENST00000632212.1""","""KI270853.1""","""retained_intron""","""ENSG00000275498.5""",[],0,"""-""",21,227,158,-206,12,149,90,-137,"""PDXDC1""","""HGNC:28995""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,false,"""other chromosome derived"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

### Filter reference chromosome

- 下流の解析では、reference chromosome (e.g., chr1, chr2...) 由来のデータを選抜する

In [9]:
down5p_no_proximal_cre_transcripts_join_ref_chr = down5p_no_proximal_cre_transcripts_join.filter(
    pl.col("seqname").str.contains("chr")
)

pl.Config.set_tbl_rows(24)
print(down5p_no_proximal_cre_transcripts_join_ref_chr.select("seqname").sort("seqname").unique())

pl.Config.set_tbl_rows(4)
print(down5p_no_proximal_cre_transcripts_join_ref_chr.select("transcript_id").n_unique())
display(down5p_no_proximal_cre_transcripts_join_ref_chr)

shape: (23, 1)
┌─────────┐
│ seqname │
│ ---     │
│ str     │
╞═════════╡
│ chr1    │
│ chr10   │
│ chr11   │
│ chr12   │
│ chr13   │
│ chr14   │
│ chr15   │
│ chr16   │
│ chr17   │
│ chr18   │
│ chr19   │
│ chr2    │
│ chr20   │
│ chr21   │
│ chr22   │
│ chr3    │
│ chr4    │
│ chr5    │
│ chr6    │
│ chr7    │
│ chr8    │
│ chr9    │
│ chrX    │
└─────────┘
4204


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32
"""ENST00000482001.1""","""chr9""","""protein_coding_CDS_not_defined""","""ENSG00000148218.17""",[],0,"""-""",80,128,198,-48,48,67,136,-19,"""ALAD""","""HGNC:395""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000609771.1""","""chr4""","""lncRNA""","""ENSG00000272885.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-""",79,127,200,-48,42,80,129,-38,null,null,"""URS00005B55C8""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000497231.7""","""chr5""","""retained_intron""","""ENSG00000164904.19""",[],0,"""-""",15,266,125,-251,10,182,59,-172,"""ALDH7A1""","""HGNC:877""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## No Proximal CRE transcirpts

- TSS ± 500 bp にない transcripts が、遺伝子レベルではどの範囲に CRE があるのか調査する

In [10]:
down5p_no_proximal_cre_transcripts_gene_cre_distance = down5p_no_proximal_cre_transcripts_join_ref_chr.join(
    human_gene_cre_association,
    on="gene_id",
    how="left"
).sort(
    "distance",
    descending=True
)


pl.Config.set_tbl_rows(8)
print(f"down5p_no_proximal_cre_transcripts_gene_cre_distance: {down5p_no_proximal_cre_transcripts_gene_cre_distance.select('transcript_id').n_unique()}")
display(down5p_no_proximal_cre_transcripts_gene_cre_distance)

down5p_no_proximal_cre_transcripts_gene_cre_distance: 4204


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str
"""ENST00000803744.1""","""chr2""","""lncRNA""","""ENSG00000230606.12""","[""TAGENE""]",1,"""-""",68,116,222,-48,43,71,137,-28,"""APPAT""","""HGNC:56009""","""URS0002A1B0DD""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null
"""ENST00000623575.4""","""chr21""","""lncRNA""","""ENSG00000275496.5""","[""TAGENE""]",1,"""-""",44,92,270,-48,24,40,187,-16,null,null,"""URS0002A5A522""","""ENSEMBL_GENCODE""",null,"""ensembl_havana_lincrna_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null
"""ENST00000576365.1""","""chr16""","""lncRNA""","""ENSG00000261997.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",26,74,306,-48,15,50,186,-35,null,null,"""URS0000197A63""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000623268.1""","""chr12""","""TEC""","""ENSG00000280088.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-""",24,72,310,-48,16,51,184,-35,null,null,null,null,"[""polya_site"", ""polya_signal""]","""havana_homo_sapiens""",null,0,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142794""",0,"""human""","""hg38""","""FCHS_142794""","""ELA"""
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142795""",0,"""human""","""hg38""","""cp5@ETFA""","""ELA"""
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142796""",0,"""human""","""hg38""","""cp1@ETFA""","""PLA"""
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142797""",0,"""human""","""hg38""","""cp4@ETFA""","""ELA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. 10 kb 範囲内に CRE が紐づかない transcript

In [11]:
no_cre_10kb_transcripts_check = down5p_no_proximal_cre_transcripts_gene_cre_distance.filter(
    pl.col("distance").is_null()
).sort(
    "HN2",
    descending=True
).with_columns(
    has_cre_gene=pl.col("gene_id").is_in(cre_gene_ids) # gene level で CRE と紐づいているかチェック
).with_columns(
    pl.lit("no gene-level CRE within 10kb").alias("flag")
)

print(f"no_cre_10kb_transcripts: {no_cre_10kb_transcripts_check.select('transcript_id').n_unique()}")
display(no_cre_10kb_transcripts_check)

no_cre_10kb_transcripts: 100


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,bool,str
"""ENST00000803744.1""","""chr2""","""lncRNA""","""ENSG00000230606.12""","[""TAGENE""]",1,"""-""",68,116,222,-48,43,71,137,-28,"""APPAT""","""HGNC:56009""","""URS0002A1B0DD""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000623575.4""","""chr21""","""lncRNA""","""ENSG00000275496.5""","[""TAGENE""]",1,"""-""",44,92,270,-48,24,40,187,-16,null,null,"""URS0002A5A522""","""ENSEMBL_GENCODE""",null,"""ensembl_havana_lincrna_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000576365.1""","""chr16""","""lncRNA""","""ENSG00000261997.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""+""",26,74,306,-48,15,50,186,-35,null,null,"""URS0000197A63""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000623268.1""","""chr12""","""TEC""","""ENSG00000280088.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]",3,"""-""",24,72,310,-48,16,51,184,-35,null,null,null,null,"[""polya_site"", ""polya_signal""]","""havana_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000469687.2""","""chr9""","""retained_intron""","""ENSG00000196873.17""",[],0,"""+""",34,174,198,-140,19,117,115,-98,"""ZNG1C""","""HGNC:18519""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000616947.2""","""chr1""","""lncRNA""","""ENSG00000290385.2""","[""nested_454_RNA_Seq_supported""]",1,"""-""",41,183,182,-142,25,119,107,-94,null,null,"""URS0000811F9F""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""nested 454 RNA-Seq supported"", ""overlaps pseudogene""]",2,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000543604.1""","""chr12""","""lncRNA""","""ENSG00000256973.1""","[""non_canonical_TEC"", ""basic"", … ""GENCODE_Primary""]",4,"""-""",34,189,183,-155,20,122,109,-102,null,null,"""URS0000300EFC""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""","[""non canonical TEC""]",1,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""
"""ENST00000524823.6""","""chr16""","""protein_coding""","""ENSG00000183889.15""","[""mRNA_start_NF"", ""cds_start_NF""]",2,"""+""",33,225,148,-192,17,142,92,-125,"""NPIPA6""","""HGNC:27893""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""no gene-level CRE within 10kb"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. 10kb 以内であれば gene level で CRE が紐づく transcripts

- distance が 0 ~ ± 500 bp のものがあるが、これは、gene level では TSS に overlap していることを示している

In [12]:
cre_10kb_transcripts_check = down5p_no_proximal_cre_transcripts_gene_cre_distance.filter(
    (pl.col("distance").is_not_null())
).sort(
    "HN2",
    descending=True
).with_columns(
    has_cre_gene=pl.col("gene_id").is_in(cre_gene_ids) # gene level で CRE と紐づいているかチェック
).with_columns(
    pl.lit("gene-level CRE within 10kb").alias("flag")
)

print(cre_10kb_transcripts_check.select("transcript_id").n_unique())
display(cre_10kb_transcripts_check)

4104


transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,bool,str
"""ENST00000488157.5""","""chr3""","""protein_coding_CDS_not_defined""","""ENSG00000010322.17""",[],0,"""+""",57,105,244,-48,42,66,143,-24,"""NISCH""","""HGNC:18006""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_286210""",9964,"""human""","""hg38""","""FCHS_286210""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000531624.1""","""chr15""","""protein_coding""","""ENSG00000171763.21""","[""mRNA_start_NF"", ""cds_start_NF"", ""exp_conf""]",3,"""+""",49,97,260,-48,27,79,145,-52,"""AFG2B""","""HGNC:28762""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_136679""",9956,"""human""","""hg38""","""FCHS_136679""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000545264.1""","""chr11""","""retained_intron""","""ENSG00000149231.17""",[],0,"""-""",36,84,286,-48,13,41,197,-28,"""CCDC82""","""HGNC:26282""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_79362""",9943,"""human""","""hg38""","""FCHS_79362""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000507792.1""","""chr17""","""retained_intron""","""ENSG00000167107.14""",[],0,"""+""",80,128,198,-48,45,84,122,-39,"""ACSF2""","""HGNC:26101""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_176922""",9919,"""human""","""hg38""","""FCHS_176922""","""ELA""",true,"""gene-level CRE within 10kb"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142794""",0,"""human""","""hg38""","""FCHS_142794""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142795""",0,"""human""","""hg38""","""cp5@ETFA""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142796""",0,"""human""","""hg38""","""cp1@ETFA""","""PLA""",true,"""gene-level CRE within 10kb"""
"""ENST00000560179.2""","""chr15""","""retained_intron""","""ENSG00000140374.18""","[""RNA_Seq_supported_partial""]",1,"""-""",11,272,123,-261,5,181,65,-176,"""ETFA""","""HGNC:3481""",null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported partial""]",1,"""FCHS_142797""",0,"""human""","""hg38""","""cp4@ETFA""","""ELA""",true,"""gene-level CRE within 10kb"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Concat

- これまで 3つの 種類のデータフレームに分けて調査してきた
- この3つそれぞれの棒グラフで、かつ transcript type ごとの 積み上げ棒グラフにする

In [13]:
all_no_proximal_cre_transcripts = pl.concat(
    [
        cre_10kb_transcripts_check,
        no_cre_10kb_transcripts_check,
        down5p_no_proximal_cre_transcripts_join_other_chr_check
    ],
    how="diagonal_relaxed" # ない列は null で埋める
)

all_no_proximal_cre_transcripts.write_parquet(
    "../../out/fantabio/no_proximal_cre_transcirpt_list/bottom5p_transcript_no_cre_v261001_validation.parquet",
    compression="zstd",
    compression_level=9
)

display(all_no_proximal_cre_transcripts)

transcript_id,seqname,transcript_type,gene_id,tags,number_of_tags,strand,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,has_cre_gene,flag
str,str,str,str,list[str],u32,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,bool,str
"""ENST00000488157.5""","""chr3""","""protein_coding_CDS_not_defined""","""ENSG00000010322.17""",[],0,"""+""",57,105,244,-48,42,66,143,-24,"""NISCH""","""HGNC:18006""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_286210""",9964,"""human""","""hg38""","""FCHS_286210""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000531624.1""","""chr15""","""protein_coding""","""ENSG00000171763.21""","[""mRNA_start_NF"", ""cds_start_NF"", ""exp_conf""]",3,"""+""",49,97,260,-48,27,79,145,-52,"""AFG2B""","""HGNC:28762""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_136679""",9956,"""human""","""hg38""","""FCHS_136679""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000545264.1""","""chr11""","""retained_intron""","""ENSG00000149231.17""",[],0,"""-""",36,84,286,-48,13,41,197,-28,"""CCDC82""","""HGNC:26282""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_79362""",9943,"""human""","""hg38""","""FCHS_79362""","""ELA""",true,"""gene-level CRE within 10kb"""
"""ENST00000507792.1""","""chr17""","""retained_intron""","""ENSG00000167107.14""",[],0,"""+""",80,128,198,-48,45,84,122,-39,"""ACSF2""","""HGNC:26101""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_176922""",9919,"""human""","""hg38""","""FCHS_176922""","""ELA""",true,"""gene-level CRE within 10kb"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000710543.1""","""KN196481.1""","""retained_intron""","""ENSG00000281500.5""",[],0,"""-""",14,200,192,-186,10,139,102,-129,"""SLC37A4""","""HGNC:4061""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""other chromosome derived"""
"""ENST00000528918.5""","""GL000251.2""","""retained_intron""","""ENSG00000227322.11""",[],0,"""-""",30,219,157,-189,18,128,105,-110,"""RXRB""","""HGNC:10478""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""other chromosome derived"""
"""ENST00000797527.1""","""GL000195.1""","""lncRNA""","""ENSG00000303853.1""","[""TAGENE""]",1,"""-""",26,226,154,-200,12,157,82,-145,null,null,"""URS0002A30CE3""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null,false,"""other chromosome derived"""
"""ENST00000632212.1""","""KI270853.1""","""retained_intron""","""ENSG00000275498.5""",[],0,"""-""",21,227,158,-206,12,149,90,-137,"""PDXDC1""","""HGNC:28995""",null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,false,"""other chromosome derived"""
